# LoRA Fine-Tuning Quick Start
This notebook demonstrates how to use the LoRA fine-tuning system for query optimization.

## 1. Setup and Imports

In [ ]:
import sys
sys.path.append('..')  # Add parent directory to path

from training.lora_trainer import LoRATrainer, quick_train
from training.model_config import get_lightweight_config, get_default_config, get_model_config
from training.data_processor import TrainingDataProcessor, convert_training_json_to_dataset

## 2. Check GPU Availability

In [ ]:
import torch

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 3. Prepare Sample Training Data

First, let's create some sample training data (or use your generated data):

In [ ]:
import json
from datetime import datetime

# Sample training data
sample_data = [
    {
        "messages": [
            {"role": "system", "content": "You are a query optimization assistant."},
            {"role": "user", "content": "Optimize this query: what are transformers?"},
            {"role": "assistant", "content": "Explain the transformer neural network architecture, including self-attention mechanisms, encoder-decoder structure, and applications in natural language processing"}
        ],
        "metadata": {"generated_at": datetime.now().isoformat()}
    },
    {
        "messages": [
            {"role": "system", "content": "You are a query optimization assistant."},
            {"role": "user", "content": "Optimize this query: how does attention work?"},
            {"role": "assistant", "content": "Describe the attention mechanism in neural networks, focusing on query-key-value computation, scaled dot-product attention, and multi-head attention implementations"}
        ],
        "metadata": {"generated_at": datetime.now().isoformat()}
    }
]

# Save sample data
sample_file = "sample_training_data.json"
with open(sample_file, 'w') as f:
    json.dump(sample_data, f, indent=2)

print(f"Created sample training data with {len(sample_data)} examples")

## 4. Process Training Data

In [ ]:
# Initialize data processor
processor = TrainingDataProcessor()

# Create dataset
dataset = processor.create_dataset(
    sample_file,
    train_test_split=0.1,
    shuffle=True
)

# Validate
stats = processor.validate_dataset(dataset['train'])
print(f"\nDataset Statistics:")
print(f"  Valid: {stats['valid']}")
print(f"  Training Examples: {stats['num_examples']}")
print(f"  Avg Text Length: {stats.get('avg_text_length', 0):.0f} chars")

# Preview first example
print(f"\nFirst Example:")
print(dataset['train'][0]['text'][:500] + "...")

## 5. Configure LoRA Training

Choose configuration based on your GPU:

In [ ]:
# Option 1: Lightweight config (8GB VRAM)
config = get_lightweight_config()

# Option 2: Default config (16GB VRAM)
# config = get_default_config()

# Option 3: Specific model
# config = get_model_config("mistral-7b")

# Customize output directory
config.output_dir = "./lora_test_model"
config.num_train_epochs = 3

print(f"Configuration:")
print(f"  Model: {config.base_model_name}")
print(f"  LoRA Rank: {config.lora_r}")
print(f"  Batch Size: {config.per_device_train_batch_size}")
print(f"  Epochs: {config.num_train_epochs}")
print(f"  Output: {config.output_dir}")

## 6. Train Model (Quick Method)

⚠️ **Warning**: This will download ~15GB model and train it. Ensure you have:
- Sufficient GPU memory
- Stable internet connection
- 20+ GB free disk space

In [ ]:
# Quick training (recommended for testing)
# Note: Requires actual training data with 100+ examples

# Uncomment to run:
# metrics = quick_train(
#     training_data_path=sample_file,
#     output_dir="./lora_test_model",
#     model_name="mistralai/Mistral-7B-Instruct-v0.2",
#     epochs=3,
#     use_lightweight=True
# )
# 
# print(f"\nTraining completed!")
# print(f"Metrics: {metrics}")

print("Training code commented out - uncomment to run actual training")

## 7. Train Model (Advanced Method)

In [ ]:
# Advanced training with custom configuration
# Uncomment to run:

# trainer = LoRATrainer(config)
# 
# # Load model and tokenizer
# trainer.load_model_and_tokenizer()
# trainer.setup_lora()
# 
# # Train
# metrics = trainer.train(
#     train_dataset=dataset['train'],
#     eval_dataset=dataset.get('test', None)
# )
# 
# print(f"\nTraining completed!")
# print(f"Final Loss: {metrics.get('train_loss', 'N/A')}")

print("Advanced training code commented out - uncomment to run")

## 8. Load and Test Trained Model

In [ ]:
# Load trained model for inference
# Uncomment after training:

# from llmservice.query_optimizer import QueryOptimizer
# 
# optimizer = QueryOptimizer()
# optimizer.load_model("./lora_test_model")
# 
# # Test queries
# test_queries = [
#     "what is LoRA?",
#     "how to fine-tune models?",
#     "explain attention mechanism"
# ]
# 
# print("\nTesting Query Optimization:\n")
# for query in test_queries:
#     optimized = optimizer.optimize_query(query)
#     print(f"Original:  {query}")
#     print(f"Optimized: {optimized}")
#     print("-" * 80)

print("Testing code commented out - uncomment after training")

## 9. Monitor Training (TensorBoard)

To view training metrics in real-time:

```bash
tensorboard --logdir ./logs/lora_training
```

Then open http://localhost:6006 in your browser.

## 10. Integration with RAG System

After training, update `config/settings.py`:

```python
USE_FINETUNED_OPTIMIZER = True
FINETUNED_OPTIMIZER_PATH = "./lora_test_model"
```

Then in your chat endpoint:

```python
from llmservice.query_optimizer import optimize_user_query

# Before retrieval
optimized_query = optimize_user_query(user_query)
results = retriever.retrieve(optimized_query, k=7)
```

## Next Steps

1. **Generate Real Data**: Upload documents and generate 500+ training examples
2. **Train Full Model**: Run training with proper dataset
3. **Evaluate**: Test on held-out queries to measure improvement
4. **Deploy**: Integrate into production RAG pipeline
5. **Monitor**: Track query optimization performance over time
6. **Iterate**: Retrain with more data as system learns user patterns